In [ ]:
import gdown
import sqlite3
import gradio as gr
from datetime import datetime

#download database
google_drive_url = 'https://drive.google.com/file/d/1fbJVLnf51FIav0zj3tYE8p7nagkCbofA/view?usp=drive_link'

def get_direct_gdrive_link(share_url):
    file_id = share_url.split('/d/')[1].split('/')[0]
    return f'https://drive.google.com/uc?id={file_id}'

def download_database(url, output='database.db'):
    direct_link = get_direct_gdrive_link(url)
    gdown.download(direct_link, output, quiet=False)
    return output

#Import database
db_file = download_database(google_drive_url)

Downloading...
From: https://drive.google.com/uc?id=1fbJVLnf51FIav0zj3tYE8p7nagkCbofA
To: /content/database.db
100%|██████████| 57.3k/57.3k [00:00<00:00, 3.56MB/s]


In [ ]:
DB_PATH = '/content/database.db'
cart = []
payment_method = ""

ALLOWED_CATEGORIES = ["Makanan berat", "Makanan ringan", "Minuman"]

# --- AUTENTIKASI ---
def cek_login(email, password):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT "ID Akun" FROM Akun WHERE "Email"=? AND "Password"=?', (email, password))
    user = cursor.fetchone()
    if not user:
        conn.close()
        return None, None
    id_akun = user[0]
    cursor.execute('SELECT 1 FROM Kios WHERE "ID Akun"=?', (id_akun,))
    if cursor.fetchone():
        conn.close()
        return id_akun, "penjual"
    cursor.execute('SELECT 1 FROM Pembeli WHERE "ID Akun"=?', (id_akun,))
    if cursor.fetchone():
        conn.close()
        return id_akun, "pembeli"
    conn.close()
    return id_akun, "tidak_diketahui"

def register_akun(email, password, role, nama, jurusan_or_kios, no_hp_or_pedagang, npm):
    try:
        # Validasi input kosong
        if not all([email, password, role, nama, no_hp_or_pedagang]):
             return "Email, Password, Role, Nama, dan No HP/Nama Pedagang harus diisi untuk melanjutkan registrasi."

        if role == "pembeli" and (not jurusan_or_kios or not npm):
             return "Jurusan dan NPM harus diisi untuk pendaftar sebagai pembeli."

        if role == "penjual" and not jurusan_or_kios:
             return "Nama Kios harus diisi untuk pendaftar sebagai penjual."

        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()

        # Cek apakah email sudah terdaftar
        cursor.execute('SELECT 1 FROM Akun WHERE Email = ?', (email,))
        if cursor.fetchone() is not None:
            conn.close()
            return "Email sudah terdaftar. Gunakan email lain atau login."

        # Insert ke tabel Akun
        cursor.execute('INSERT INTO Akun (Email, Password) VALUES (?, ?)', (email, password))
        id_akun = cursor.lastrowid

        if role == "pembeli":
            cursor.execute('INSERT INTO Pembeli (NPM, Nama, Jurusan, "Nomor Telepon", "ID Akun") VALUES (?, ?, ?, ?, ?)',
                           (npm, nama, jurusan_or_kios, no_hp_or_pedagang, id_akun))
        elif role == "penjual":
            cursor.execute('SELECT MAX("ID Kios") FROM Kios')
            last_id_kios = cursor.fetchone()[0]
            new_id_kios = 100 if last_id_kios is None else last_id_kios + 1
            cursor.execute('INSERT INTO Kios ("ID Kios", "Nama Kios", "Nama Pedagang", "ID Akun") VALUES (?, ?, ?, ?)',
                           (new_id_kios, jurusan_or_kios, no_hp_or_pedagang, id_akun))
        else:
            conn.close()
            return "Role tidak valid."

        conn.commit()
        conn.close()
        return "Registrasi berhasil. Silakan login."
    except Exception as e:
        return f"Error saat registrasi: {e}"


# --- HALAMAN PEMBELI ---
def ambil_profil_pembeli(id_akun):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT NPM, Nama, Jurusan, "Nomor Telepon" FROM Pembeli WHERE "ID Akun"=?', (int(id_akun),))
    data = cursor.fetchone()
    conn.close()
    if not data:
        return "Data profil tidak ditemukan."
    npm, nama, jurusan, telp = data
    return f"NPM: {npm}\nNama: {nama}\nJurusan: {jurusan}\nNomor Telepon: {telp}"

def ubah_profil_pembeli(id_akun, email_baru, password_baru, telp_baru):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    try:
        if email_baru:
            cursor.execute('UPDATE Akun SET Email=? WHERE "ID Akun"=?', (email_baru, int(id_akun)))
        if password_baru:
            cursor.execute('UPDATE Akun SET Password=? WHERE "ID Akun"=?', (password_baru, int(id_akun)))
        if telp_baru:
            cursor.execute('UPDATE Pembeli SET "Nomor Telepon"=? WHERE "ID Akun"=?', (telp_baru, int(id_akun)))

        conn.commit()
        return "Profil berhasil diperbarui."
    except Exception as e:
        return f"Gagal memperbarui profil: {e}"
    finally:
        conn.close()

# --- MENU DAN KIOS ---
def get_list_kios():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT "ID Kios", "Nama Kios" FROM Kios')
    kios = cursor.fetchall()
    conn.close()
    return {nama: id_kios for id_kios, nama in kios}

def get_menu_by_kios(id_kios):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT "ID Menu", "Menu" FROM Menu WHERE "ID Kios"=?', (id_kios,))
    menu = cursor.fetchall()
    conn.close()
    return {nama: id_menu for id_menu, nama in menu}

def get_kiosk_name_by_id(id_kios):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT "Nama Kios" FROM Kios WHERE "ID Kios"=?', (id_kios,))
    result = cursor.fetchone()
    conn.close()
    return result[0] if result else "Kios Tidak Ditemukan"

# --- CART ---
def tambah_ke_cart(nama_menu):
    cart.append(nama_menu)
    return f"Ditambahkan ke cart: {nama_menu}"

def lihat_cart():
    if not cart:
        return "Cart kosong."
    return "\n".join(cart)

def kosongkan_cart():
    cart.clear()
    return "Cart dikosongkan."

def simpan_metode(metode):
    global payment_method
    payment_method = metode
    return f"Metode pembayaran: {metode}"

def konfirmasi_order(hidden_id):
    if not cart:
        return "Cart kosong, tidak bisa konfirmasi."
    if not payment_method:
        return "Silakan pilih metode pembayaran terlebih dahulu."

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    try:
        cursor.execute('SELECT NPM FROM Pembeli WHERE "ID Akun"=?', (int(hidden_id),))
        row = cursor.fetchone()
        if not row:
            return "Gagal ambil NPM pembeli."
        npm = row[0]

        today_str = datetime.today().strftime('%Y-%m-%d')
        hari = today_str[-2:]
        bulan = today_str[5:7]
        kode_tanggal = f"{bulan}{hari}"
        cursor.execute("SELECT COUNT(DISTINCT 'ID Transaksi') FROM 'Transaksi Penjualan' WHERE Tanggal = ?", (today_str,))
        urutan_hari_ini = cursor.fetchone()[0] + 1
        id_transaksi = f"{kode_tanggal}{str(urutan_hari_ini).zfill(3)}"

        for nama_menu in cart:
            cursor.execute('SELECT "ID Menu" FROM Menu WHERE "Menu"=?', (nama_menu,))
            row = cursor.fetchone()
            if not row:
                continue
            id_menu = row[0]

            cursor.execute("""
                INSERT INTO 'Transaksi Penjualan' (
                    "ID Transaksi", "Tanggal", "NPM", "ID Menu", "Jumlah", "Metode Pembayaran"
                ) VALUES (?, ?, ?, ?, ?, ?)
            """, (id_transaksi, today_str, npm, id_menu, 1, payment_method))

        conn.commit()
        return f"🎉 Order berhasil dicatat dengan ID Transaksi: {id_transaksi}"
    except Exception as e:
        return f"❌ Error: {e}"
    finally:
        conn.close()
        cart.clear()

# --- HALAMAN PENJUAL ---
def get_id_kios_from_akun(id_akun):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT "ID Kios" FROM Kios WHERE "ID Akun"=?', (id_akun,))
    result = cursor.fetchone()
    conn.close()
    return result[0] if result else None

def lihat_profil_kios(id_akun):
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute('SELECT "ID Kios", "Nama Kios", "Nama Pedagang", "ID Akun" FROM Kios WHERE "ID Akun"=?', (int(id_akun),))
        data = cursor.fetchone()
        conn.close()
        if not data:
            return "Profil kios tidak ditemukan."
        id_kios, nama_kios, nama_pedagang, id_akun = data
        return f"ID Kios: {id_kios}\nNama Kios: {nama_kios}\nNama Pedagang: {nama_pedagang}\nID Akun: {id_akun}"
    except Exception as e:
        return f"Error: {e}"

def lihat_semua_menu_penjual(id_akun):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    id_kios = get_id_kios_from_akun(id_akun)
    if not id_kios:
        conn.close()
        return "Tidak dapat menemukan ID Kios untuk akun ini."

    query = '''
        SELECT
            Menu."ID Menu",
            Menu."Menu",
            Menu.Kategori,
            Menu.Harga,
            Kios."Nama Kios",
            Kios."Nama Pedagang"
        FROM Menu
        INNER JOIN Kios ON Menu."ID Kios" = Kios."ID Kios"
        WHERE Menu."ID Kios" = ?
    '''
    cursor.execute(query, (id_kios,)) # Pass id_kios as a parameter
    data = cursor.fetchall()
    conn.close()

    return "\n".join([
        f"{id_menu} - {menu} - {kategori} - Rp{harga} - {nama_kios} (oleh {nama_pedagang})"
        for id_menu, menu, kategori, harga, nama_kios, nama_pedagang in data
    ]) or "Tidak ada menu yang terdaftar untuk kios Anda."

def lihat_semua_kios():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT * FROM Kios')
    data = cursor.fetchall()
    conn.close()
    return "\n".join([str(row) for row in data]) or "Data kosong."

def lihat_trx_penjual(id_akun):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    id_kios = get_id_kios_from_akun(id_akun)
    if not id_kios:
        conn.close()
        return "Tidak dapat menemukan ID Kios untuk akun ini."

    query = '''
        SELECT
            TP."ID Transaksi",
            TP.Tanggal,
            TP.NPM,
            M.Menu,
            TP.Jumlah,
            TP."Metode Pembayaran"
        FROM "Transaksi Penjualan" AS TP
        INNER JOIN Menu AS M ON TP."ID Menu" = M."ID Menu"
        WHERE M."ID Kios" = ?
    '''
    cursor.execute(query, (id_kios,))
    data = cursor.fetchall()
    conn.close()
    return "\n".join([str(row) for row in data]) or "Tidak ada transaksi untuk kios Anda."

# === GRADIO APP ===
with gr.Blocks() as app:
    gr.Markdown("## Kantin Terintegrasi - Login")

    with gr.Row():
        email_in = gr.Textbox(label="Email")
        password_in = gr.Textbox(label="Password", type="password")
        login_btn = gr.Button("Login")

    status_out = gr.Textbox(label="Status", interactive=False)

    with gr.Accordion("Belum punya akun? Daftar di sini", open=False):
        role_reg = gr.Dropdown(choices=["pembeli", "penjual"], label="Daftar sebagai")
        email_reg = gr.Textbox(label="Email")
        password_reg = gr.Textbox(label="Password", type="password")
        nama_reg = gr.Textbox(label="Nama")
        npm_reg = gr.Textbox(label="NPM", visible=False)
        field1_reg = gr.Textbox(label="", visible=False) # Jurusan or Nama Kios
        field2_reg = gr.Textbox(label="", visible=False) # No HP or Nama Pedagang
        reg_btn = gr.Button("Daftar")
        reg_status = gr.Textbox(label="Status Registrasi")

        def update_form_fields(role):
            if role == "pembeli":
                return gr.update(visible=True), gr.update(label="Jurusan", visible=True), gr.update(label="No HP", visible=True)
            elif role == "penjual":
                return gr.update(visible=False), gr.update(label="Nama Kios", visible=True), gr.update(label="Nama Pedagang", visible=True)
            return gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)

        role_reg.change(fn=update_form_fields, inputs=role_reg, outputs=[npm_reg, field1_reg, field2_reg])

        reg_btn.click(register_akun,
                      inputs=[email_reg, password_reg, role_reg, nama_reg, field1_reg, field2_reg, npm_reg],
                      outputs=reg_status)
    hidden_id = gr.Textbox(visible=False, value="0")

    pembeli_box = gr.Column(visible=False)
    with pembeli_box:
        gr.Markdown("### Halaman Pembeli")

        pembeli_main_menu = gr.Column(visible=True)
        with pembeli_main_menu:
            btn_buka_profil = gr.Button("Lihat Profil")
            btn_buka_order = gr.Button("Order")

        profil_box = gr.Column(visible=False)
        with profil_box:
            btn_back_profil = gr.Button("← Back")
            out_profil = gr.Textbox(label="Profil", lines=4, interactive=False)
            btn_lihat_profil = gr.Button("Refresh Profil")
            email_baru = gr.Textbox(label="Email Baru")
            password_baru = gr.Textbox(label="Password Baru", type="password")
            telp_baru = gr.Textbox(label="Nomor Telepon Baru")
            btn_update_profil = gr.Button("Update Profil")
            out_update = gr.Textbox(label="Status Update", lines=2, interactive=False)

        order_box = gr.Column(visible=False)
        with order_box:
            btn_back_order = gr.Button("← Back")
            dropdown_kios = gr.Dropdown(label="Pilih Kios", choices=list(get_list_kios().keys()))
            dropdown_menu = gr.Dropdown(label="Pilih Menu", choices=[])

            btn_refresh_kios = gr.Button("Refresh Kios")

            def refresh_kios_dropdown():
              kios_dict = get_list_kios()
              return gr.update(choices=list(kios_dict.keys()), value=None), gr.update(choices=[], value=None)

            btn_refresh_kios.click(fn=refresh_kios_dropdown, outputs=[dropdown_kios, dropdown_menu])

            def update_menu_dropdown(nama_kios):
                kios_dict = get_list_kios()
                id_kios = kios_dict.get(nama_kios)
                if not id_kios:
                    return gr.update(choices=[], value=None)
                menu_dict = get_menu_by_kios(id_kios)
                return gr.update(choices=list(menu_dict.keys()), value=None)

            dropdown_kios.change(fn=update_menu_dropdown, inputs=dropdown_kios, outputs=dropdown_menu)

            out_cart = gr.Textbox(label="Isi Cart", lines=6)

            btn_tambah_cart = gr.Button("Tambah ke Cart")
            out_cart_msg = gr.Textbox(label="Status Cart", lines=2)

            def tambah_ke_cart_dan_lihat(nama_menu):
              msg = tambah_ke_cart(nama_menu)
              return msg, lihat_cart()

            btn_tambah_cart.click(fn=tambah_ke_cart_dan_lihat, inputs=dropdown_menu, outputs=[out_cart_msg, out_cart])

            btn_kosongkan = gr.Button("Kosongkan Cart")

            def kosongkan_cart_dan_lihat():
              msg = kosongkan_cart()
              return msg, lihat_cart()

            btn_kosongkan.click(fn=kosongkan_cart_dan_lihat, outputs=[out_cart_msg, out_cart])


            dropdown_metode = gr.Dropdown(label="Pilih Metode Pembayaran", choices=["Tunai", "Non Tunai"])
            out_metode = gr.Textbox(label="Metode Dipilih", lines=1)
            dropdown_metode.change(fn=simpan_metode, inputs=dropdown_metode, outputs=out_metode)

            btn_konfirmasi = gr.Button("Confirm Order")
            out_confirm = gr.Textbox(label="Status Order")
            btn_konfirmasi.click(fn=konfirmasi_order, inputs=hidden_id, outputs=out_confirm)

        def buka_profil():
            return gr.update(visible=False), gr.update(visible=True)

        def tutup_profil():
            return gr.update(visible=True), gr.update(visible=False)

        def buka_order():
            return gr.update(visible=False), gr.update(visible=True)

        def tutup_order():
            return gr.update(visible=True), gr.update(visible=False)

        btn_buka_profil.click(fn=buka_profil, outputs=[pembeli_main_menu, profil_box])
        btn_back_profil.click(fn=tutup_profil, outputs=[pembeli_main_menu, profil_box])
        btn_lihat_profil.click(fn=ambil_profil_pembeli, inputs=hidden_id, outputs=out_profil)
        btn_update_profil.click(fn=ubah_profil_pembeli, inputs=[hidden_id, email_baru, password_baru, telp_baru], outputs=out_update)

        btn_buka_order.click(fn=buka_order, outputs=[pembeli_main_menu, order_box])
        btn_back_order.click(fn=tutup_order, outputs=[pembeli_main_menu, order_box])

    penjual_box = gr.Column(visible=False)
    with penjual_box:
        gr.Markdown("### Halaman Penjual")

        with gr.Tab("Lihat Data"):
            btn_menu_all = gr.Button("Lihat Menu Saya")
            btn_kios_all = gr.Button("Lihat Semua Kios")
            btn_trx_all = gr.Button("Lihat Transaksi Saya")
            out_lihat = gr.Textbox(label="Hasil", lines=15)

            btn_menu_all.click(fn=lihat_semua_menu_penjual, inputs=hidden_id, outputs=out_lihat)
            btn_kios_all.click(fn=lihat_semua_kios, outputs=out_lihat)
            btn_trx_all.click(fn=lihat_trx_penjual, inputs=hidden_id, outputs=out_lihat)

        with gr.Tab("Tambah Menu"):
            nama_add_menu = gr.Textbox(label="Nama Menu")
            kategori_add_menu = gr.Dropdown(label="Kategori", choices=ALLOWED_CATEGORIES)
            harga_add_menu = gr.Textbox(label="Harga")
            btn_add_menu = gr.Button("Tambah Menu")
            out_add_menu = gr.Textbox(label="Status")

            def tambah_menu(nama, kategori, harga, id_akun):
                try:
                    if kategori not in ALLOWED_CATEGORIES:
                        return "Kategori tidak valid. Pilih dari daftar yang tersedia."

                    id_kios = get_id_kios_from_akun(int(id_akun))
                    if not id_kios:
                        return "ID Kios tidak ditemukan."

                    conn = sqlite3.connect(DB_PATH)
                    cursor = conn.cursor()

                    prefix = (id_kios % 100) * 1000
                    cursor.execute('SELECT MAX("ID Menu") FROM Menu WHERE "ID Menu" BETWEEN ? AND ?', (prefix, prefix + 999))
                    max_id = cursor.fetchone()[0]
                    next_id = prefix + 1 if max_id is None else max_id + 1

                    cursor.execute('INSERT INTO Menu ("ID Menu", "Menu", Kategori, Harga, "ID Kios") VALUES (?, ?, ?, ?, ?)',
                                   (next_id, nama, kategori, int(harga), id_kios))
                    conn.commit()
                    conn.close()
                    return f"Menu berhasil ditambahkan dengan ID {next_id}."
                except ValueError:
                    return "Harga harus berupa angka."
                except Exception as e:
                    return f"Error: {e}"

            btn_add_menu.click(tambah_menu, inputs=[nama_add_menu, kategori_add_menu, harga_add_menu, hidden_id], outputs=out_add_menu)

        with gr.Tab("Profil Kios"):
            btn_lihat_profil_kios = gr.Button("Lihat Profil Saya")
            out_profil_kios = gr.Textbox(label="Profil Kios", lines=5)
            btn_lihat_profil_kios.click(fn=lihat_profil_kios, inputs=[hidden_id], outputs=out_profil_kios)

        with gr.Tab("Edit Menu"):
            id_menu_edit = gr.Textbox(label="ID Menu")
            nama_edit = gr.Textbox(label="Nama Menu Baru")
            kategori_edit = gr.Dropdown(label="Kategori Baru", choices=ALLOWED_CATEGORIES)
            harga_edit = gr.Textbox(label="Harga Baru")
            btn_update_menu = gr.Button("Update Menu")
            out_update_menu = gr.Textbox(label="Status")

            def update_menu(id_menu, nama, kategori, harga, id_akun):
                try:
                    if kategori not in ALLOWED_CATEGORIES:
                        return "Kategori tidak valid. Pilih dari daftar yang tersedia."

                    id_kios = get_id_kios_from_akun(int(id_akun))
                    conn = sqlite3.connect(DB_PATH)
                    cursor = conn.cursor()
                    cursor.execute('SELECT "ID Kios" FROM Menu WHERE "ID Menu"=?', (int(id_menu),))
                    menu_kios = cursor.fetchone()
                    if not menu_kios or menu_kios[0] != id_kios:
                        conn.close()
                        return "Anda tidak memiliki akses untuk mengedit menu ini."
                    cursor.execute('UPDATE Menu SET "Menu"=?, Kategori=?, Harga=? WHERE "ID Menu"=?',
                                   (nama, kategori, int(harga), int(id_menu)))
                    conn.commit()
                    conn.close()
                    return "Menu berhasil diperbarui."
                except ValueError:
                    return "Harga harus berupa angka."
                except Exception as e:
                    return f"Error: {e}"

            btn_update_menu.click(update_menu, inputs=[id_menu_edit, nama_edit, kategori_edit, harga_edit, hidden_id], outputs=out_update_menu)

        with gr.Tab("Hapus Menu"):
            id_menu_hapus = gr.Textbox(label="ID Menu")
            btn_hapus_menu = gr.Button("Hapus Menu")
            out_hapus_menu = gr.Textbox(label="Status")

            def hapus_menu(id_menu, id_akun):
                try:
                    id_kios = get_id_kios_from_akun(int(id_akun))
                    conn = sqlite3.connect(DB_PATH)
                    cursor = conn.cursor()
                    cursor.execute('SELECT "ID Kios" FROM Menu WHERE "ID Menu"=?', (int(id_menu),))
                    menu_kios = cursor.fetchone()
                    if not menu_kios or menu_kios[0] != id_kios:
                        conn.close()
                        return "Anda tidak memiliki akses untuk menghapus menu ini."
                    cursor.execute('DELETE FROM Menu WHERE "ID Menu"=?', (int(id_menu),))
                    conn.commit()
                    conn.close()
                    return "Menu berhasil dihapus."
                except Exception as e:
                    return f"Error: {e}"

            btn_hapus_menu.click(hapus_menu, inputs=[id_menu_hapus, hidden_id], outputs=out_hapus_menu)

        with gr.Tab("Kelola Kios"):
            nama_kios_kelola = gr.Textbox(label="Nama Kios Baru")
            nama_pedagang_kelola = gr.Textbox(label="Nama Pedagang Baru")
            email_baru_akun = gr.Textbox(label="Email Akun Baru") # Renamed to avoid conflict
            password_baru_akun = gr.Textbox(label="Password Akun Baru", type="password") # Renamed to avoid conflict
            btn_update_kios = gr.Button("Update Kios dan Akun")
            out_kios = gr.Textbox(label="Status")

            def update_kios_dan_akun(nama_kios, nama_pedagang, email_baru, password_baru, id_akun):
                try:
                    # Validasi input tidak kosong
                    for field_name, value in {
                        "Nama Kios": nama_kios,
                        "Nama Pedagang": nama_pedagang,
                        "Email Baru": email_baru,
                        "Password Baru": password_baru,
                        "ID Akun": id_akun
                    }.items():
                        if not str(value).strip():
                            return f"Field '{field_name}' tidak boleh kosong."

                    id_akun = int(id_akun)
                    id_kios = get_id_kios_from_akun(id_akun)
                    if not id_kios:
                        return "ID Kios tidak ditemukan."

                    conn = sqlite3.connect(DB_PATH)
                    cursor = conn.cursor()

                    # Cek apakah email baru sudah dipakai oleh akun lain
                    cursor.execute(
                        'SELECT 1 FROM Akun WHERE "Email"=? AND "ID Akun"!=?',
                        (email_baru.strip(), id_akun)
                    )
                    if cursor.fetchone():
                        conn.close()
                        return "Email sudah digunakan oleh akun lain."

                    # Cek apakah nama kios sudah dipakai oleh kios lain
                    cursor.execute(
                        'SELECT 1 FROM Kios WHERE "Nama Kios"=? AND "ID Kios"!=?',
                        (nama_kios.strip(), id_kios)
                    )
                    if cursor.fetchone():
                        conn.close()
                        return "Nama kios sudah digunakan oleh kios lain."

                    # Lanjutkan update karena valid
                    cursor.execute(
                        'UPDATE Kios SET "Nama Kios"=?, "Nama Pedagang"=? WHERE "ID Kios"=?',
                        (nama_kios.strip(), nama_pedagang.strip(), id_kios)
                    )

                    cursor.execute(
                        'UPDATE Akun SET "Email"=?, "Password"=? WHERE "ID Akun"=?',
                        (email_baru.strip(), password_baru.strip(), id_akun)
                    )

                    conn.commit()
                    conn.close()
                    return "Kios dan akun berhasil diperbarui."

                except Exception as e:
                    return f"Error: {e}"


            btn_update_kios.click(fn=update_kios_dan_akun,
                                  inputs=[nama_kios_kelola, nama_pedagang_kelola, email_baru_akun, password_baru_akun, hidden_id],
                                  outputs=out_kios)


    def proses_login(email, password):
        id_akun, role = cek_login(email, password)
        if not id_akun:
            return "Email atau password salah.", gr.update(visible=False), gr.update(visible=False), ""
        if role == "penjual":
            return f"Login berhasil sebagai PENJUAL (ID: {id_akun})", gr.update(visible=False), gr.update(visible=True), str(id_akun)
        elif role == "pembeli":
            return f"Login berhasil sebagai PEMBELI (ID: {id_akun})", gr.update(visible=True), gr.update(visible=False), str(id_akun)
        else:
            return "Akun tidak dikenali.", gr.update(visible=False), gr.update(visible=False), ""

    login_btn.click(fn=proses_login,
                    inputs=[email_in, password_in],
                    outputs=[status_out, pembeli_box, penjual_box, hidden_id])

app.launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://962d985dbae038e5e4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
